# Needleman-Wunsch sequence disntance

Implementation and meaning exploration

In [43]:
import numpy as np

def needleman_wunsch(seq1, seq2, 
                     match=5, mismatch=-4, gap=-8,
                     ends_free=False):
    """
    Needleman-Wunsch global sequence alignment.
    
    Parameters
    ----------
    seq1, seq2 : str
        Input sequences (DNA or protein)
    match : int
        Score for matching bases (default 5)
    mismatch : int  
        Penalty for mismatching bases (default -4)
    gap : int
        Penalty for gap opening (default -8)
    ends_free : bool
        If True, terminal gaps are not penalized (semi-global alignment)
    
    Returns
    -------
    score : float
        Alignment score
    aligned_seq1 : str
        Aligned version of seq1 (with gaps as '-')
    aligned_seq2 : str
        Aligned version of seq2 (with gaps as '-')
    matrix : np.ndarray
        The filled scoring matrix
    """
    n, m = len(seq1), len(seq2)
    
    # initialize scoring matrix and traceback matrix
    score_matrix = np.zeros((n+1, m+1))
    traceback    = np.zeros((n+1, m+1), dtype=int)  # 0=stop, 1=diag, 2=up, 3=left
    
    # fill first row and column
    for i in range(1, n+1):
        score_matrix[i, 0] = 0 if ends_free else gap * i
        traceback[i, 0] = 2  # up
    for j in range(1, m+1):
        score_matrix[0, j] = 0 if ends_free else gap * j
        traceback[0, j] = 3  # left

    # fill the matrix
    zeroindexed=True
    if not zeroindexed:
        for i in range(1, n+1):
            for j in range(1, m+1):

                match_score = match if seq1[i-1] == seq2[j-1] else mismatch

                diag = score_matrix[i-1, j-1] + match_score
                up   = score_matrix[i-1, j  ] + gap
                left = score_matrix[i,   j-1] + gap

                best = max(diag, up, left)
                score_matrix[i, j] = best

                if   best == diag: traceback[i, j] = 1
                elif best == up:   traceback[i, j] = 2
                else:              traceback[i, j] = 3
    else:
        for i in range(0, n):
            for j in range(0, m):
                
                match_score = match if seq1[i] == seq2[j] else mismatch
                
                # tree ways in which I can arrive at score_matrix[i+1,j+1]
                diag = score_matrix[i  , j  ] + match_score # diagnal movement (add to precursor score the match)
                up   = score_matrix[i  , j+1] + gap         # horizontal move  (add to precursor score the gap penalty)
                left = score_matrix[i+1, j  ] + gap         # vertical   move  (add to precursor score the gap penalty)
                
                best = max(diag, up, left)
                score_matrix[i+1, j+1] = best
                
                if   best == diag: traceback[i+1, j+1] = 1
                elif best == up:   traceback[i+1, j+1] = 2
                else:              traceback[i+1, j+1] = 3

    # find start of traceback
    if ends_free:
        # find best score in last row or last column
        last_row_max = np.max(score_matrix[n, :])
        last_col_max = np.max(score_matrix[:, m])
        
        if last_row_max >= last_col_max:
            j = np.argmax(score_matrix[n, :])
            i = n
        else:
            i = np.argmax(score_matrix[:, m])
            j = m
        final_score = score_matrix[i, j]
    else:
        i, j = n, m
        final_score = score_matrix[n, m]

    # traceback
    aligned1, aligned2 = [], []
    
    while i > 0 or j > 0:
        if ends_free and i == 0 and j == 0:
            break
        
        if i == 0:
            aligned1.append('-')
            aligned2.append(seq2[j-1])
            j -= 1
        elif j == 0:
            aligned1.append(seq1[i-1])
            aligned2.append('-')
            i -= 1
        elif traceback[i, j] == 1:  # diagonal
            aligned1.append(seq1[i-1])
            aligned2.append(seq2[j-1])
            i -= 1; j -= 1
        elif traceback[i, j] == 2:  # up
            aligned1.append(seq1[i-1])
            aligned2.append('-')
            i -= 1
        else:                        # left
            aligned1.append('-')
            aligned2.append(seq2[j-1])
            j -= 1

    aligned1 = ''.join(reversed(aligned1))
    aligned2 = ''.join(reversed(aligned2))

    return final_score, aligned1, aligned2, score_matrix,traceback

In [113]:
seq1 = "ACAAATGCAT"[::-1]
seq2 = "BAATGC"[::-1]

In [114]:
score,alseq1,alseq2,score_matrix,traceback = needleman_wunsch(seq1,seq2,ends_free=False)

In [115]:
score

np.float64(-11.0)

In [116]:
traceback

array([[0, 3, 3, 3, 3, 3, 3],
       [2, 1, 1, 1, 3, 3, 3],
       [2, 1, 1, 1, 1, 1, 3],
       [2, 1, 1, 1, 2, 1, 1],
       [2, 2, 1, 3, 1, 1, 1],
       [2, 2, 2, 1, 3, 3, 1],
       [2, 2, 2, 2, 1, 1, 3],
       [2, 2, 2, 2, 1, 1, 3],
       [2, 2, 2, 2, 1, 1, 1],
       [2, 1, 2, 2, 2, 2, 1],
       [2, 2, 2, 2, 1, 1, 1]])

In [117]:
pd.DataFrame(score_matrix,index=list("-"+seq1),columns=list("-"+seq2))

,-,C,G,T,A,A,B
-,0.0,-8.0,-16.0,-24.0,-32.0,-40.0,-48.0
T,-8.0,-4.0,-12.0,-11.0,-19.0,-27.0,-35.0
A,-16.0,-12.0,-8.0,-16.0,-6.0,-14.0,-22.0
C,-24.0,-11.0,-16.0,-12.0,-14.0,-10.0,-18.0
G,-32.0,-19.0,-6.0,-14.0,-16.0,-18.0,-14.0
T,-40.0,-27.0,-14.0,-1.0,-9.0,-17.0,-22.0
A,-48.0,-35.0,-22.0,-9.0,4.0,-4.0,-12.0
A,-56.0,-43.0,-30.0,-17.0,-4.0,9.0,1.0
A,-64.0,-51.0,-38.0,-25.0,-12.0,1.0,5.0
C,-72.0,-59.0,-46.0,-33.0,-20.0,-7.0,-3.0


In [118]:
print(seq1)
print(seq2)
print(alseq1)
print(alseq2)

TACGTAAACA
CGTAAB
TACGTAAACA
--CGT-AA-B
